In [ ]:
# ============================================================================
# SETUP: Ollama + ngrok para acceso remoto
# ============================================================================
# IMPORTANTE: Este cell instala Ollama y configura ngrok con el flag correcto
# para evitar errores 403 al conectarse remotamente.

!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Instalar dependencias de Python
!pip install -q ollama pyngrok

# Configurar ngrok (necesitas tu token de ngrok.com)
from google.colab import userdata
import os

try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    !ngrok config add-authtoken {NGROK_AUTH_TOKEN}
    print("✅ ngrok configurado correctamente")
except Exception as e:
    print(f"⚠️ No se encontró NGROK_AUTH_TOKEN en Colab secrets: {e}")
    print("   Puedes configurarlo en: Secrets (🔑) en el panel izquierdo")

# Iniciar Ollama en segundo plano
import subprocess
import threading
import time

def run_ollama():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

thread = threading.Thread(target=run_ollama, daemon=True)
thread.start()
time.sleep(5)
print("✅ Servidor Ollama iniciado en localhost:11434")

# Iniciar ngrok con el flag --host-header para evitar 403
# CRÍTICO: El flag --host-header="localhost:11434" es necesario para que
# Ollama acepte las peticiones que vienen a través del túnel ngrok
import asyncio

async def start_ngrok():
    proc = await asyncio.create_subprocess_exec(
        'ngrok', 'http', '--log', 'stderr', '11434', 
        '--host-header=localhost:11434',  # ← FIX: Evita error 403
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE
    )
    
    # Esperar a que ngrok inicie
    await asyncio.sleep(3)
    
    # Obtener la URL pública de ngrok
    import requests
    try:
        response = requests.get('http://localhost:4040/api/tunnels')
        tunnels = response.json()['tunnels']
        public_url = tunnels[0]['public_url']
        print(f"✅ ngrok túnel activo: {public_url}")
        print(f"   Puedes usar esta URL para conectarte remotamente")
        return public_url
    except Exception as e:
        print(f"⚠️ No se pudo obtener la URL de ngrok: {e}")
        return None

# Ejecutar ngrok
try:
    ngrok_url = await start_ngrok()
    
    # Guardar la URL para uso posterior
    if ngrok_url:
        os.environ['OLLAMA_HOST_REMOTE'] = ngrok_url
        print(f"\n📝 Variable de entorno configurada: OLLAMA_HOST_REMOTE={ngrok_url}")
except Exception as e:
    print(f"⚠️ Error al iniciar ngrok: {e}")
    print("   Puedes continuar usando Ollama localmente")

print("\n" + "="*60)
print("SETUP COMPLETADO")
print("="*60)
print("Ollama está corriendo y accesible vía ngrok")
print("Ahora puedes ejecutar el siguiente cell para descargar un modelo")


In [ ]:
# ============================================================================
# Descargar modelo de Ollama (opcional)
# ============================================================================
# Descomentar para descargar un modelo específico

# !ollama pull gemma2:9b
# !ollama pull llama3.2:3b
# !ollama pull qwen2.5:7b

print("Modelos disponibles:")
!ollama list


In [ ]:
# ============================================================================
# Instalar dependencias de Python
# ============================================================================

!pip install -q pandas numpy scikit-learn torch transformers accelerate
!pip install -q sentence-transformers huggingface_hub python-dotenv
!pip install -q imbalanced-learn xgboost lightgbm
!pip install -q ollama

print("✅ Dependencias instaladas")


In [ ]:
# -*- coding: utf-8 -*-
"""
WomenHelp-MX 2026 — Clasificación de Reportes de Violencia de Género
======================================================================
Subtask 1: Clasificación multi-clase (4 niveles de severidad)
Subtask 2: Clasificación multi-label (7 tipos de violencia)

Fase final: predicciones sobre test.csv (sin etiquetas).
Los archivos de test NO tienen cabecera ni columna ID; cada línea es un texto.
"""

from __future__ import annotations
import json, os, warnings, zipfile, re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# Scikit-learn
from sklearn.metrics import classification_report, f1_score, hamming_loss

# Hugging Face
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
)
from huggingface_hub import InferenceClient

# Ollama (opcional)
try:
    import ollama
    OLLAMA_AVAILABLE = True
except ImportError:
    OLLAMA_AVAILABLE = False

# Imbalanced learning (opcional)
try:
    from imblearn.over_sampling import SMOTE
    IMBLEARN_AVAILABLE = True
except ImportError:
    IMBLEARN_AVAILABLE = False

warnings.filterwarnings("ignore")

# ===========================================================================
# CONFIGURACIÓN PRINCIPAL
# ===========================================================================

# Método de clasificación:
#   "hf_local"            – Fine-tuning de modelo HF en local
#   "hf_api"              – Hugging Face Inference API (prompting)
#   "ollama_local"        – Ollama corriendo en localhost
#   "ollama_remote"       – Ollama en host remoto (tu servidor)
#   "ollama_cloud"        – Ollama Cloud (modelos cloud de ollama.com)
#   "embedding_classifier"– Sentence Transformers + clasificador clásico (recomendado)
METHOD = "embedding_classifier"

# ---------- Hugging Face ----------
HF_LOCAL_MODEL  = "PlanTL-GOB-ES/roberta-base-bne"
HF_API_MODEL    = "mistralai/Mistral-7B-Instruct-v0.3"
HF_API_TOKEN    = os.getenv("HF_TOKEN", "")

# ---------- Ollama ----------
OLLAMA_MODEL       = "gemma2:9b"
OLLAMA_HOST_LOCAL  = "http://localhost:11434"
OLLAMA_HOST_REMOTE = os.getenv("OLLAMA_HOST", "http://localhost:11434")

# ---------- Ollama Cloud (ollama.com) ----------
OLLAMA_CLOUD_HOST  = "https://ollama.com"
OLLAMA_CLOUD_MODEL = "gpt-oss:120b"  # Modelo cloud de Ollama
OLLAMA_API_KEY     = os.getenv("OLLAMA_API_KEY", "")  # API key de ollama.com

# ---------- Embeddings ----------
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

# ---------- Entrenamiento ----------
SEED         = 42
BATCH_SIZE   = 16
MAX_LENGTH   = 512
NUM_EPOCHS   = 5
LEARNING_RATE = 2e-5

# ---------- Clases ----------
SEVERITY_LABELS = ["Mild", "Medium", "High", "Severe"]
VIOLENCE_TYPES  = ["Economic", "Physical", "Patrimonial",
                   "Psychological", "Sexual", "Vicarious", "N/A"]

# ---------- Rutas de datos ----------
# Fase de desarrollo (con etiquetas, para evaluar)
SUBTASK1_TRAIN = "subtask1/train.csv"
SUBTASK1_DEV   = "subtask1/devel.csv"
SUBTASK2_TRAIN = "subtask2/train.csv"
SUBTASK2_DEV   = "subtask2/devel.csv"

# Fase final: archivos de test SIN cabecera, SIN etiquetas
# Cada línea es un texto (puede estar entre comillas CSV).
SUBTASK1_TEST  = "testingkit_extracted/subtask1/test.csv"
SUBTASK2_TEST  = "testingkit_extracted/subtask2/test.csv"

# Modo de ejecución:
#   "dev"  – entrena en train, evalúa en devel (útil para desarrollo)
#   "test" – entrena en train+devel, predice en test (para submission final)
RUN_MODE = "test"

OUTPUT_DIR = Path("predictions")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- Device ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Method: {METHOD}")
print(f"Run mode: {RUN_MODE}")


# ===========================================================================
# CARGA Y PREPROCESAMIENTO DE DATOS
# ===========================================================================

def load_subtask1_data(
    train_path: str,
    eval_path: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Carga datos de Subtask 1.
    Ambos archivos tienen cabecera: ID, TEXT, CLASS
    """
    df_train = pd.read_csv(train_path, encoding="utf-8")
    df_eval  = pd.read_csv(eval_path,  encoding="utf-8")

    print(f"Subtask 1 — Train: {len(df_train)} | Eval: {len(df_eval)}")
    print(f"  Distribución de clases (train):\n{df_train['CLASS'].value_counts().sort_index()}")
    return df_train, df_eval


def load_subtask1_test(test_path: str) -> pd.DataFrame:
    """
    Carga el archivo de test de Subtask 1.
    Sin cabecera, sin ID, sin etiqueta — solo texto (una fila por documento).
    """
    # El archivo puede tener comillas CSV alrededor de los textos
    df = pd.read_csv(test_path, header=None, names=["TEXT"],
                     encoding="utf-8", quoting=0, on_bad_lines="skip")
    print(f"Subtask 1 — Test: {len(df)} registros")
    return df


def load_subtask2_data(
    train_path: str,
    eval_path: str,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Carga datos de Subtask 2.
    Ambos archivos tienen cabecera: ID, Text, L0..L6
    """
    df_train = pd.read_csv(train_path, encoding="utf-8")
    df_eval  = pd.read_csv(eval_path,  encoding="utf-8")

    label_cols = [f"L{i}" for i in range(7)]
    print(f"Subtask 2 — Train: {len(df_train)} | Eval: {len(df_eval)}")
    print(f"  Distribución de labels (train):\n{df_train[label_cols].sum()}")
    return df_train, df_eval


def load_subtask2_test(test_path: str) -> pd.DataFrame:
    """
    Carga el archivo de test de Subtask 2.
    Sin cabecera, sin ID, sin etiquetas — solo texto.
    """
    df = pd.read_csv(test_path, header=None, names=["Text"],
                     encoding="utf-8", quoting=0, on_bad_lines="skip")
    print(f"Subtask 2 — Test: {len(df)} registros")
    return df


def preprocess_text(text: str) -> str:
    """Limpieza básica de texto."""
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


# ===========================================================================
# CLASIFICADORES — SUBTASK 1
# ===========================================================================

class HF_Local_Classifier:
    """Fine-tuning de un modelo HF para clasificación de severidad."""

    def __init__(self, model_name: str = HF_LOCAL_MODEL, num_labels: int = 4):
        print(f"Cargando modelo HF Local: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
            device_map="auto" if device == "cuda" else None,
        )
        self.model_name = model_name

    def train(self, texts: List[str], labels: List[int], epochs: int = NUM_EPOCHS):
        from transformers import TrainingArguments, Trainer

        encodings = self.tokenizer(
            texts, truncation=True, padding=True, max_length=MAX_LENGTH
        )

        class _Dataset(torch.utils.data.Dataset):
            def __init__(self, enc, lbl):
                self.enc = enc
                self.lbl = lbl
            def __getitem__(self, idx):
                item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
                item["labels"] = torch.tensor(self.lbl[idx])
                return item
            def __len__(self):
                return len(self.lbl)

        dataset = _Dataset(encodings, labels)
        args = TrainingArguments(
            output_dir="./results",
            num_train_epochs=epochs,
            per_device_train_batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            weight_decay=0.01,
            logging_steps=50,
            save_strategy="no",
            report_to="none",
        )
        trainer = Trainer(model=self.model, args=args, train_dataset=dataset)
        print(f"Entrenando {epochs} epochs...")
        trainer.train()

    def predict(self, texts: List[str]) -> List[int]:
        predictions = []
        for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="HF Local predict"):
            batch = texts[i : i + BATCH_SIZE]
            enc = self.tokenizer(
                batch, truncation=True, padding=True,
                max_length=MAX_LENGTH, return_tensors="pt"
            ).to(self.model.device)
            with torch.no_grad():
                logits = self.model(**enc).logits
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                predictions.extend(preds.tolist())
        return predictions


class HF_API_Classifier:
    """Clasificador de severidad vía Hugging Face Inference API."""

    def __init__(self, model_name: str = HF_API_MODEL, token: str = HF_API_TOKEN):
        print(f"Conectando a HF Inference API: {model_name}")
        self.client = InferenceClient(model=model_name, token=token or None)
        self.model_name = model_name

    def _classify_severity(self, text: str) -> int:
        prompt = (
            "Eres un clasificador experto en violencia de género en México.\n"
            "Clasifica el siguiente reporte en uno de 4 niveles de severidad:\n\n"
            "0: Mild (Leve) - Sin agresión física, conflictos menores\n"
            "1: Medium (Medio) - Amenazas, control, violencia psicológica moderada\n"
            "2: High (Alto) - Agresión física, amenazas graves, control extremo\n"
            "3: Severe (Severo) - Peligro de vida, agresión sexual, violencia extrema\n\n"
            f"REPORTE:\n{text[:1500]}\n\n"
            "Responde SOLO con el número (0, 1, 2, o 3):"
        )
        try:
            response = self.client.text_generation(
                prompt, max_new_tokens=5, temperature=0.1, do_sample=False
            )
            numbers = re.findall(r"[0-3]", response)
            return int(numbers[0]) if numbers else 1
        except Exception as e:
            print(f"Error HF API: {e}")
            return 1

    def predict(self, texts: List[str]) -> List[int]:
        return [self._classify_severity(t) for t in tqdm(texts, desc="HF API")]


class Ollama_Classifier:
    """Clasificador de severidad vía Ollama (local, remoto o cloud)."""

    def __init__(self, model_name: str = OLLAMA_MODEL, host: str = OLLAMA_HOST_LOCAL, 
                 use_cloud: bool = False, api_key: str = ""):
        """
        Inicializa el clasificador Ollama.
        
        Args:
            model_name: Nombre del modelo (ej: "gemma2:9b", "gpt-oss:120b")
            host: URL del servidor Ollama (local, remoto o cloud)
            use_cloud: Si True, usa Ollama Cloud (ollama.com)
            api_key: API key para Ollama Cloud (opcional)
        """
        self.model_name = model_name
        self.use_cloud = use_cloud
        
        if use_cloud:
            print(f"Conectando a Ollama Cloud: {model_name} @ {host}")
            # Para Ollama Cloud, agregar header de autenticación
            headers = {}
            if api_key:
                headers['Authorization'] = f'Bearer {api_key}'
            self.client = ollama.Client(host=host, headers=headers)
        else:
            print(f"Conectando a Ollama: {model_name} @ {host}")
            self.client = ollama.Client(host=host)
        
        try:
            # Verificar conexión
            if not use_cloud:
                self.client.list()
            print("✅ Conexión a Ollama establecida")
        except Exception as e:
            print(f"⚠️ Error conectando a Ollama: {e}")

    def _classify_severity(self, text: str) -> int:
        """
        Clasifica la severidad de un reporte de violencia.
        Prompt optimizado basado en investigación de clasificación de violencia.
        """
        # Prompt mejorado con contexto específico de México y ejemplos implícitos
        prompt = (
            "Eres un experto en clasificación de violencia de género en México, "
            "entrenado para evaluar reportes de mujeres víctimas de violencia.\n\n"
            
            "Tu tarea es clasificar el siguiente reporte en UNO de estos 4 niveles de severidad:\n\n"
            
            "**0 - LEVE (Mild):**\n"
            "- Conflictos verbales sin amenazas\n"
            "- Discusiones sin violencia física\n"
            "- Tensión en la relación sin agresión\n"
            "- Celos o control leve\n\n"
            
            "**1 - MEDIO (Medium):**\n"
            "- Amenazas verbales directas\n"
            "- Control económico o de movimientos\n"
            "- Violencia psicológica moderada (insultos, humillaciones)\n"
            "- Intimidación sin violencia física\n"
            "- Aislamiento social\n\n"
            
            "**2 - ALTO (High):**\n"
            "- Agresión física (golpes, empujones, jalones)\n"
            "- Amenazas graves de muerte o daño\n"
            "- Control extremo (encierro, vigilancia constante)\n"
            "- Violencia psicológica severa\n"
            "- Destrucción de propiedades\n"
            "- Violencia vicaria (amenazas a hijos)\n\n"
            
            "**3 - SEVERO (Severe):**\n"
            "- Peligro inminente de muerte\n"
            "- Agresión sexual o violación\n"
            "- Violencia física extrema (armas, estrangulamiento, quemaduras)\n"
            "- Intento de feminicidio\n"
            "- Secuestro o privación de libertad\n"
            "- Lesiones graves que requieren atención médica urgente\n\n"
            
            "**INSTRUCCIONES:**\n"
            "1. Lee cuidadosamente el reporte completo\n"
            "2. Identifica los indicadores de violencia presentes\n"
            "3. Evalúa el nivel de peligro y daño\n"
            "4. Responde ÚNICAMENTE con el número (0, 1, 2 o 3)\n"
            "5. NO agregues explicaciones, solo el número\n\n"
            
            f"**REPORTE A CLASIFICAR:**\n{text[:1500]}\n\n"
            
            "**TU CLASIFICACIÓN (solo el número):**"
        )
        
        try:
            resp = self.client.generate(
                model=self.model_name,
                prompt=prompt,
                options={
                    "temperature": 0.1,      # Baja temperatura para respuestas consistentes
                    "num_predict": 10,       # Permitir un poco más de tokens
                    "top_p": 0.9,            # Nucleus sampling
                    "top_k": 40,             # Top-k sampling
                    "repeat_penalty": 1.1,   # Evitar repeticiones
                },
            )
            
            # Extraer el número de la respuesta
            response_text = resp["response"].strip()
            numbers = re.findall(r"[0-3]", response_text)
            
            if numbers:
                return int(numbers[0])
            else:
                # Si no encuentra número, intentar clasificar por palabras clave
                response_lower = response_text.lower()
                if "severe" in response_lower or "severo" in response_lower:
                    return 3
                elif "high" in response_lower or "alto" in response_lower:
                    return 2
                elif "medium" in response_lower or "medio" in response_lower:
                    return 1
                else:
                    return 0  # Default a leve si no hay claridad
                    
        except Exception as e:
            print(f"⚠️ Error en clasificación: {e}")
            return 1  # Default a medio en caso de error

    def predict(self, texts: List[str]) -> List[int]:
        """Predice la severidad para una lista de textos."""
        return [self._classify_severity(t) for t in tqdm(texts, desc="Ollama Severity")]


class EmbeddingClassifier:
    """Embeddings + StackingClassifier para clasificación de severidad."""

    def __init__(self, embedding_model: str = EMBEDDING_MODEL, num_labels: int = 4):
        print(f"Cargando modelo de embeddings: {embedding_model}")
        from sentence_transformers import SentenceTransformer
        self.emb_model = SentenceTransformer(
            embedding_model, trust_remote_code=True, device=device
        )
        self.num_labels = num_labels
        self.classifier = None

    def _encode(self, texts: List[str]) -> np.ndarray:
        return self.emb_model.encode(
            texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

    def train(self, texts: List[str], labels: List[int]):
        from sklearn.ensemble import StackingClassifier, RandomForestClassifier
        from sklearn.linear_model import LogisticRegression
        from xgboost import XGBClassifier
        from lightgbm import LGBMClassifier

        print("Generando embeddings (train)...")
        embeddings = self._encode(texts)

        if IMBLEARN_AVAILABLE and len(set(labels)) > 1:
            print("Aplicando SMOTE...")
            try:
                smote = SMOTE(random_state=SEED, k_neighbors=3)
                embeddings, labels = smote.fit_resample(embeddings, labels)
                print(f"  Tras SMOTE: {len(embeddings)} muestras")
            except Exception as e:
                print(f"SMOTE falló: {e}")

        estimators = [
            ("rf",   RandomForestClassifier(n_estimators=200, max_depth=10,
                                            class_weight="balanced",
                                            random_state=SEED, n_jobs=-1)),
            ("xgb",  XGBClassifier(n_estimators=200, max_depth=6,
                                   learning_rate=0.1,
                                   random_state=SEED, n_jobs=-1,
                                   eval_metric="mlogloss")),
            ("lgbm", LGBMClassifier(n_estimators=200, max_depth=8,
                                    learning_rate=0.05,
                                    class_weight="balanced",
                                    random_state=SEED, n_jobs=-1, verbose=-1)),
        ]
        self.classifier = StackingClassifier(
            estimators=estimators,
            final_estimator=LogisticRegression(
                C=1.0, class_weight="balanced", random_state=SEED
            ),
            cv=5,
            n_jobs=-1,
        )
        print("Entrenando clasificador...")
        self.classifier.fit(embeddings, labels)

    def predict(self, texts: List[str]) -> List[int]:
        print("Generando embeddings (predict)...")
        embeddings = self._encode(texts)
        return self.classifier.predict(embeddings).tolist()


def get_subtask1_classifier(method: str = METHOD):
    if method == "hf_local":
        return HF_Local_Classifier()
    elif method == "hf_api":
        return HF_API_Classifier()
    elif method == "ollama_local":
        return Ollama_Classifier(host=OLLAMA_HOST_LOCAL)
    elif method == "ollama_remote":
        return Ollama_Classifier(host=OLLAMA_HOST_REMOTE)
    elif method == "ollama_cloud":
        return Ollama_Classifier(
            model_name=OLLAMA_CLOUD_MODEL,
            host=OLLAMA_CLOUD_HOST,
            use_cloud=True,
            api_key=OLLAMA_API_KEY
        )
    elif method == "embedding_classifier":
        return EmbeddingClassifier()
    else:
        raise ValueError(f"Método no reconocido: {method}")


# ===========================================================================
# CLASIFICADORES — SUBTASK 2 (multi-label)
# ===========================================================================

class MultiLabelEmbeddingClassifier:
    """Embeddings + un clasificador logístico binario por cada label."""

    def __init__(self, embedding_model: str = EMBEDDING_MODEL, num_labels: int = 7):
        print(f"Cargando modelo de embeddings: {embedding_model}")
        from sentence_transformers import SentenceTransformer
        self.emb_model = SentenceTransformer(
            embedding_model, trust_remote_code=True, device=device
        )
        self.num_labels = num_labels
        self.classifiers: List = []

    def _encode(self, texts: List[str]) -> np.ndarray:
        return self.emb_model.encode(
            texts,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

    def train(self, texts: List[str], labels: np.ndarray):
        from sklearn.linear_model import LogisticRegression

        print("Generando embeddings (train)...")
        embeddings = self._encode(texts)

        print("Entrenando clasificadores binarios por label...")
        self.classifiers = []
        for i in tqdm(range(self.num_labels), desc="Labels"):
            clf = LogisticRegression(
                class_weight="balanced", max_iter=1000,
                random_state=SEED, C=1.0
            )
            clf.fit(embeddings, labels[:, i])
            self.classifiers.append(clf)

    def predict(self, texts: List[str], threshold: float = 0.5) -> np.ndarray:
        print("Generando embeddings (predict)...")
        embeddings = self._encode(texts)
        preds = np.zeros((len(texts), self.num_labels), dtype=int)
        for i, clf in enumerate(self.classifiers):
            probs = clf.predict_proba(embeddings)[:, 1]
            preds[:, i] = (probs >= threshold).astype(int)

        # Regla de consistencia: si todos los tipos de violencia son 0, forzar N/A=1
        # y si algún tipo de violencia es 1, forzar N/A=0
        for row in range(len(preds)):
            if preds[row, :6].sum() == 0:
                preds[row, 6] = 1   # N/A
            else:
                preds[row, 6] = 0   # no N/A si hay violencia detectada

        return preds


class MultiLabelLLMClassifier:
    """Clasificador multi-label usando LLM con prompting optimizado."""

    def __init__(self, method: str = "hf_api", model_name: str = HF_API_MODEL, 
                 use_cloud: bool = False, api_key: str = ""):
        """
        Inicializa el clasificador multi-label.
        
        Args:
            method: Método de clasificación ("hf_api", "ollama_local", "ollama_remote", "ollama_cloud")
            model_name: Nombre del modelo
            use_cloud: Si True, usa Ollama Cloud
            api_key: API key para Ollama Cloud
        """
        self.method = method
        self.model_name = model_name
        self.use_cloud = use_cloud
        
        if method == "hf_api":
            self.client = InferenceClient(
                model=model_name, token=HF_API_TOKEN or None
            )
        elif method in ("ollama_local", "ollama_remote", "ollama_cloud"):
            if method == "ollama_cloud" or use_cloud:
                host = OLLAMA_CLOUD_HOST
                headers = {}
                if api_key:
                    headers['Authorization'] = f'Bearer {api_key}'
                self.client = ollama.Client(host=host, headers=headers)
            else:
                host = OLLAMA_HOST_LOCAL if method == "ollama_local" else OLLAMA_HOST_REMOTE
                self.client = ollama.Client(host=host)
        else:
            self.client = None

    def _classify_types(self, text: str) -> List[int]:
        """
        Clasifica los tipos de violencia presentes en un reporte.
        Prompt optimizado con ejemplos y contexto específico.
        """
        # Prompt mejorado con descripciones detalladas y ejemplos
        prompt = (
            "Eres un experto en clasificación de violencia de género en México, "
            "especializado en identificar múltiples tipos de violencia que pueden coexistir.\n\n"
            
            "**TIPOS DE VIOLENCIA A IDENTIFICAR:**\n\n"
            
            "**0 - ECONÓMICA (Economic):**\n"
            "- Control del dinero o ingresos de la víctima\n"
            "- Prohibición de trabajar o estudiar\n"
            "- Limitación del acceso a recursos económicos\n"
            "- Obligación a entregar salario o pensión\n"
            "- Destrucción de herramientas de trabajo\n"
            "Ejemplos: 'no me deja trabajar', 'me quita mi dinero', 'no me da para gastos'\n\n"
            
            "**1 - FÍSICA (Physical):**\n"
            "- Golpes, empujones, jalones de cabello\n"
            "- Uso de objetos o armas para agredir\n"
            "- Quemaduras, mordidas, estrangulamiento\n"
            "- Cualquier contacto físico violento\n"
            "Ejemplos: 'me pegó', 'me empujó', 'me jaló del cabello', 'me golpeó con'\n\n"
            
            "**2 - PATRIMONIAL (Patrimonial):**\n"
            "- Destrucción de objetos personales o del hogar\n"
            "- Daño a documentos importantes (IDs, títulos)\n"
            "- Robo o retención de bienes\n"
            "- Destrucción de ropa, celular, muebles\n"
            "Ejemplos: 'rompió mi celular', 'destruyó mis cosas', 'me quitó mis documentos'\n\n"
            
            "**3 - PSICOLÓGICA (Psychological):**\n"
            "- Insultos, humillaciones, desprecios\n"
            "- Amenazas verbales (no de muerte)\n"
            "- Control de movimientos, vestimenta, amistades\n"
            "- Celos excesivos, vigilancia constante\n"
            "- Aislamiento social\n"
            "- Manipulación emocional, chantaje\n"
            "Ejemplos: 'me insulta', 'me humilla', 'me controla', 'no me deja salir', 'revisa mi celular'\n\n"
            
            "**4 - SEXUAL (Sexual):**\n"
            "- Violación o intento de violación\n"
            "- Acoso sexual, tocamientos no consentidos\n"
            "- Obligación a tener relaciones sexuales\n"
            "- Coerción sexual, chantaje sexual\n"
            "- Exhibicionismo forzado\n"
            "Ejemplos: 'me obligó a', 'me violó', 'me tocó sin mi consentimiento', 'me forzó'\n\n"
            
            "**5 - VICARIA (Vicarious):**\n"
            "- Violencia ejercida a través de los hijos\n"
            "- Amenazas de quitar a los hijos\n"
            "- Uso de los hijos para dañar a la madre\n"
            "- Maltrato a los hijos para afectar a la madre\n"
            "- Manipulación de los hijos contra la madre\n"
            "Ejemplos: 'amenaza con quitarme a mis hijos', 'les dice cosas malas de mí', 'maltrata a los niños'\n\n"
            
            "**6 - N/A (Sin información suficiente):**\n"
            "- El reporte no describe violencia clara\n"
            "- Información insuficiente para clasificar\n"
            "- Solo si NINGUNO de los otros tipos está presente\n\n"
            
            "**INSTRUCCIONES:**\n"
            "1. Lee el reporte completo cuidadosamente\n"
            "2. Para CADA tipo de violencia (0-6), determina si está presente (1) o no (0)\n"
            "3. PUEDEN coexistir múltiples tipos (ej: física + psicológica + económica)\n"
            "4. Si identificas AL MENOS UN tipo de violencia (0-5), entonces N/A debe ser 0\n"
            "5. Solo marca N/A=1 si NO hay ningún tipo de violencia identificable\n"
            "6. Responde ÚNICAMENTE con 7 números separados por comas\n"
            "7. Formato: Economic,Physical,Patrimonial,Psychological,Sexual,Vicarious,N/A\n"
            "8. Ejemplo de respuesta: 1,1,0,1,0,0,0 (económica + física + psicológica)\n\n"
            
            f"**REPORTE A CLASIFICAR:**\n{text[:1500]}\n\n"
            
            "**TU CLASIFICACIÓN (7 números separados por comas):**"
        )
        
        try:
            if self.method == "hf_api":
                response = self.client.text_generation(
                    prompt, 
                    max_new_tokens=30, 
                    temperature=0.1, 
                    do_sample=False
                )
            else:
                response = self.client.generate(
                    model=self.model_name,
                    prompt=prompt,
                    options={
                        "temperature": 0.1,
                        "num_predict": 30,
                        "top_p": 0.9,
                        "top_k": 40,
                        "repeat_penalty": 1.1,
                    },
                )["response"]

            # Extraer números de la respuesta
            numbers = re.findall(r"[01]", response)
            
            if len(numbers) >= 7:
                labels = [int(n) for n in numbers[:7]]
                
                # Aplicar regla de consistencia N/A
                if sum(labels[:6]) > 0:  # Si hay algún tipo de violencia
                    labels[6] = 0  # N/A debe ser 0
                else:  # Si no hay violencia
                    labels[6] = 1  # N/A debe ser 1
                
                return labels
            else:
                # Si no se pueden extraer números, retornar N/A
                return [0, 0, 0, 0, 0, 0, 1]
                
        except Exception as e:
            print(f"⚠️ Error en clasificación multi-label: {e}")
            return [0, 0, 0, 0, 0, 0, 1]  # Default a N/A en caso de error

    def predict(self, texts: List[str]) -> np.ndarray:
        """Predice los tipos de violencia para una lista de textos."""
        return np.array(
            [self._classify_types(t) for t in tqdm(texts, desc="Multi-label LLM")]
        )


# ===========================================================================
# EVALUACIÓN
# ===========================================================================

def evaluate_subtask1(y_true: List[int], y_pred: List[int]) -> Dict[str, float]:
    macro_f1    = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    print("\n" + "=" * 50)
    print("SUBTASK 1 — Evaluación")
    print("=" * 50)
    print(f"Macro-F1:    {macro_f1:.4f}")
    print(f"Weighted-F1: {weighted_f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=SEVERITY_LABELS))

    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1}


def evaluate_subtask2(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    macro_f1    = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    micro_f1    = f1_score(y_true, y_pred, average="micro",    zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    h_loss      = hamming_loss(y_true, y_pred)

    print("\n" + "=" * 50)
    print("SUBTASK 2 — Evaluación")
    print("=" * 50)
    print(f"Macro-F1:    {macro_f1:.4f}")
    print(f"Micro-F1:    {micro_f1:.4f}")
    print(f"Weighted-F1: {weighted_f1:.4f}")
    print(f"Hamming Loss:{h_loss:.4f}")

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "weighted_f1": weighted_f1,
        "hamming_loss": h_loss,
    }


# ===========================================================================
# GENERACIÓN DE ARCHIVOS DE SUBMISSION
# ===========================================================================

def generate_submission_subtask1(predictions: List[int], output_path: Path):
    """
    Formato requerido: un entero por línea, sin cabecera.
    El número de línea corresponde al mismo índice en el test.csv.
    """
    with open(output_path, "w", encoding="utf-8") as f:
        for pred in predictions:
            f.write(f"{pred}\n")
    print(f"Subtask 1 guardado: {output_path} ({len(predictions)} predicciones)")


def generate_submission_subtask2(predictions: np.ndarray, output_path: Path):
    """
    Formato requerido: vector binario separado por comas, sin cabecera.
    Ej: 1,0,1,1,1,0,0
    """
    with open(output_path, "w", encoding="utf-8") as f:
        for pred in predictions:
            f.write(",".join(map(str, pred.astype(int))) + "\n")
    print(f"Subtask 2 guardado: {output_path} ({len(predictions)} predicciones)")


def create_submission_zip(
    subtask1_path: Optional[Path],
    subtask2_path: Optional[Path],
    zip_path: Path,
):
    """
    Crea el ZIP de submission.
    Los CSV deben estar en la raíz del ZIP (sin carpetas).
    """
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        if subtask1_path and subtask1_path.exists():
            zf.write(subtask1_path, arcname="subtask1.csv")
        if subtask2_path and subtask2_path.exists():
            zf.write(subtask2_path, arcname="subtask2.csv")
    print(f"ZIP creado: {zip_path}")


# ===========================================================================
# PIPELINES DE EJECUCIÓN
# ===========================================================================

def run_subtask1(
    df_train: pd.DataFrame,
    df_eval: Optional[pd.DataFrame],
    df_test: Optional[pd.DataFrame],
    mode: str = RUN_MODE,
) -> Tuple[List[int], Dict[str, float]]:
    """
    Pipeline Subtask 1.

    mode="dev"  → entrena en train, evalúa en devel (df_eval debe tener CLASS)
    mode="test" → entrena en train+devel, predice en test (sin etiquetas)
    """
    print("\n" + "=" * 60)
    print("SUBTASK 1 — Clasificación de Severidad")
    print("=" * 60)

    classifier = get_subtask1_classifier(METHOD)

    if mode == "dev":
        assert df_eval is not None, "df_eval requerido en modo dev"
        texts_train  = df_train["TEXT"].apply(preprocess_text).tolist()
        labels_train = df_train["CLASS"].tolist()
        texts_eval   = df_eval["TEXT"].apply(preprocess_text).tolist()
        labels_eval  = df_eval["CLASS"].tolist()

        if hasattr(classifier, "train"):
            classifier.train(texts_train, labels_train)

        print("Prediciendo en conjunto de desarrollo...")
        predictions = classifier.predict(texts_eval)
        metrics = evaluate_subtask1(labels_eval, predictions)
        return predictions, metrics

    else:  # mode == "test"
        assert df_test is not None, "df_test requerido en modo test"

        # Combinar train + devel para entrenar con todos los datos etiquetados
        if df_eval is not None and "CLASS" in df_eval.columns:
            df_all = pd.concat([df_train, df_eval], ignore_index=True)
            print(f"Entrenando con train+devel: {len(df_all)} registros")
        else:
            df_all = df_train
            print(f"Entrenando solo con train: {len(df_all)} registros")

        texts_all  = df_all["TEXT"].apply(preprocess_text).tolist()
        labels_all = df_all["CLASS"].tolist()
        texts_test = df_test["TEXT"].apply(preprocess_text).tolist()

        if hasattr(classifier, "train"):
            classifier.train(texts_all, labels_all)

        print("Prediciendo en conjunto de test...")
        predictions = classifier.predict(texts_test)
        return predictions, {}


def run_subtask2(
    df_train: pd.DataFrame,
    df_eval: Optional[pd.DataFrame],
    df_test: Optional[pd.DataFrame],
    mode: str = RUN_MODE,
) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Pipeline Subtask 2.

    mode="dev"  → entrena en train, evalúa en devel
    mode="test" → entrena en train+devel, predice en test
    """
    print("\n" + "=" * 60)
    print("SUBTASK 2 — Clasificación de Tipos de Violencia")
    print("=" * 60)

    label_cols = [f"L{i}" for i in range(7)]

    # Seleccionar clasificador
    if METHOD in ("hf_api", "ollama_local", "ollama_remote", "ollama_cloud"):
        # Normalizar método para el clasificador
        if METHOD == "hf_api":
            clf_method = "hf_api"
        elif METHOD == "ollama_cloud":
            clf_method = "ollama_cloud"
            classifier = MultiLabelLLMClassifier(
                method=clf_method,
                model_name=OLLAMA_CLOUD_MODEL,
                use_cloud=True,
                api_key=OLLAMA_API_KEY
            )
        else:
            clf_method = "ollama_local" if METHOD == "ollama_local" else "ollama_remote"
        
        if METHOD != "ollama_cloud":
            classifier = MultiLabelLLMClassifier(method=clf_method)
    else:
        classifier = MultiLabelEmbeddingClassifier()

    if mode == "dev":
        assert df_eval is not None, "df_eval requerido en modo dev"
        texts_train  = df_train["Text"].apply(preprocess_text).tolist()
        labels_train = df_train[label_cols].values
        texts_eval   = df_eval["Text"].apply(preprocess_text).tolist()

        if hasattr(classifier, "train"):
            classifier.train(texts_train, labels_train)

        print("Prediciendo en conjunto de desarrollo...")
        predictions = classifier.predict(texts_eval)

        if all(c in df_eval.columns for c in label_cols):
            labels_eval = df_eval[label_cols].values
            metrics = evaluate_subtask2(labels_eval, predictions)
        else:
            metrics = {}
            print("Sin ground truth para evaluación")

        return predictions, metrics

    else:  # mode == "test"
        assert df_test is not None, "df_test requerido en modo test"

        if df_eval is not None and all(c in df_eval.columns for c in label_cols):
            df_all = pd.concat([df_train, df_eval], ignore_index=True)
            print(f"Entrenando con train+devel: {len(df_all)} registros")
        else:
            df_all = df_train
            print(f"Entrenando solo con train: {len(df_all)} registros")

        texts_all  = df_all["Text"].apply(preprocess_text).tolist()
        labels_all = df_all[label_cols].values
        texts_test = df_test["Text"].apply(preprocess_text).tolist()

        if hasattr(classifier, "train"):
            classifier.train(texts_all, labels_all)

        print("Prediciendo en conjunto de test...")
        predictions = classifier.predict(texts_test)
        return predictions, {}


# ===========================================================================
# EJECUCIÓN PRINCIPAL
# ===========================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("WomenHelp-MX 2026 — Pipeline de Clasificación")
    print(f"Modo: {RUN_MODE.upper()}")
    print("=" * 60)

    metrics1: Dict[str, float] = {}
    metrics2: Dict[str, float] = {}
    subtask1_path: Optional[Path] = None
    subtask2_path: Optional[Path] = None

    # -----------------------------------------------------------------------
    # SUBTASK 1
    # -----------------------------------------------------------------------
    if os.path.exists(SUBTASK1_TRAIN):
        df1_train, df1_dev = load_subtask1_data(SUBTASK1_TRAIN, SUBTASK1_DEV)

        if RUN_MODE == "test":
            if os.path.exists(SUBTASK1_TEST):
                df1_test = load_subtask1_test(SUBTASK1_TEST)
            else:
                print(f"ADVERTENCIA: No se encontró {SUBTASK1_TEST}. "
                      "Usando devel como test (sin evaluación).")
                df1_test = df1_dev[["TEXT"]].copy()
        else:
            df1_test = None

        pred1, metrics1 = run_subtask1(
            df_train=df1_train,
            df_eval=df1_dev,
            df_test=df1_test,
            mode=RUN_MODE,
        )

        subtask1_path = OUTPUT_DIR / "subtask1.csv"
        generate_submission_subtask1(pred1, subtask1_path)
    else:
        print(f"ADVERTENCIA: No se encontró {SUBTASK1_TRAIN}. Saltando Subtask 1.")

    # -----------------------------------------------------------------------
    # SUBTASK 2
    # -----------------------------------------------------------------------
    if os.path.exists(SUBTASK2_TRAIN):
        df2_train, df2_dev = load_subtask2_data(SUBTASK2_TRAIN, SUBTASK2_DEV)

        if RUN_MODE == "test":
            if os.path.exists(SUBTASK2_TEST):
                df2_test = load_subtask2_test(SUBTASK2_TEST)
            else:
                print(f"ADVERTENCIA: No se encontró {SUBTASK2_TEST}. "
                      "Usando devel como test (sin evaluación).")
                df2_test = df2_dev[["Text"]].copy()
        else:
            df2_test = None

        pred2, metrics2 = run_subtask2(
            df_train=df2_train,
            df_eval=df2_dev,
            df_test=df2_test,
            mode=RUN_MODE,
        )

        subtask2_path = OUTPUT_DIR / "subtask2.csv"
        generate_submission_subtask2(pred2, subtask2_path)
    else:
        print(f"ADVERTENCIA: No se encontró {SUBTASK2_TRAIN}. Saltando Subtask 2.")

    # -----------------------------------------------------------------------
    # ZIP de submission
    # -----------------------------------------------------------------------
    if subtask1_path or subtask2_path:
        zip_path = OUTPUT_DIR / "predictions.zip"
        create_submission_zip(subtask1_path, subtask2_path, zip_path)

    # -----------------------------------------------------------------------
    # Guardar métricas
    # -----------------------------------------------------------------------
    metrics_path = OUTPUT_DIR / "metrics.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(
            {"method": METHOD, "mode": RUN_MODE,
             "subtask1": metrics1, "subtask2": metrics2},
            f, indent=2, ensure_ascii=False,
        )
    print(f"\nMétricas guardadas: {metrics_path}")
    print("\n" + "=" * 60)
    print("Pipeline completado")
    print("=" * 60)


In [ ]:
# ============================================================================
# CONFIGURACIÓN PRINCIPAL
# ============================================================================
# Ajusta estos parámetros según tus necesidades

# ============================================================================
# MÉTODO DE CLASIFICACIÓN
# ============================================================================
# Opciones disponibles:
#   "embedding_classifier" - Recomendado para competencia (rápido, sin GPU)
#   "hf_local"            - Fine-tuning local (requiere GPU ≥8GB)
#   "hf_api"              - Hugging Face API (requiere token)
#   "ollama_local"        - Ollama en localhost
#   "ollama_remote"       - Ollama en servidor remoto (tu servidor)
#   "ollama_cloud"        - Ollama Cloud (modelos cloud de ollama.com)

METHOD = "embedding_classifier"  # ✅ Recomendado

# ============================================================================
# CONFIGURACIÓN DE OLLAMA
# ============================================================================

# ---------- Ollama Local ----------
OLLAMA_MODEL = "gemma2:9b"
OLLAMA_HOST_LOCAL = "http://localhost:11434"

# ---------- Ollama Remoto (tu servidor propio) ----------
import os
OLLAMA_HOST_REMOTE = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
# Ejemplo: OLLAMA_HOST_REMOTE = "http://192.168.1.100:11434"
# O con dominio: OLLAMA_HOST_REMOTE = "https://ollama.tu-dominio.com"

# ---------- Ollama Cloud (ollama.com) ----------
OLLAMA_CLOUD_HOST = "https://ollama.com"
OLLAMA_CLOUD_MODEL = "gpt-oss:120b"  # Modelo cloud de Ollama
OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY', '')

# Para configurar Ollama Cloud:
# 1. Crea cuenta en ollama.com
# 2. Genera API key en ollama.com/settings/api
# 3. Configura en Colab Secrets: OLLAMA_API_KEY

# ============================================================================
# OTRAS CONFIGURACIONES
# ============================================================================

# Modo de ejecución
RUN_MODE = "test"  # "dev" para desarrollo, "test" para submission final

# Modelo de embeddings (para METHOD="embedding_classifier")
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

# ============================================================================
# VERIFICACIÓN DE CONFIGURACIÓN
# ============================================================================

print("="*60)
print("CONFIGURACIÓN ACTUAL")
print("="*60)
print(f"Método: {METHOD}")
print(f"Modo: {RUN_MODE}")

if METHOD == "embedding_classifier":
    print(f"Modelo de embeddings: {EMBEDDING_MODEL}")
elif METHOD == "ollama_local":
    print(f"Ollama local: {OLLAMA_HOST_LOCAL}")
    print(f"Modelo: {OLLAMA_MODEL}")
elif METHOD == "ollama_remote":
    print(f"Ollama remoto: {OLLAMA_HOST_REMOTE}")
    print(f"Modelo: {OLLAMA_MODEL}")
elif METHOD == "ollama_cloud":
    print(f"Ollama Cloud: {OLLAMA_CLOUD_HOST}")
    print(f"Modelo: {OLLAMA_CLOUD_MODEL}")
    if OLLAMA_API_KEY:
        print(f"✅ API Key configurada")
    else:
        print(f"⚠️ API Key NO configurada - configura OLLAMA_API_KEY en Secrets")

print("="*60)

# ============================================================================
# EJEMPLO: Configurar Ollama Cloud
# ============================================================================
# from google.colab import userdata
# OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')
# METHOD = "ollama_cloud"

# ============================================================================
# EJEMPLO: Configurar Ollama Remoto
# ============================================================================
# OLLAMA_HOST_REMOTE = "http://192.168.1.100:11434"  # Tu servidor
# METHOD = "ollama_remote"


In [ ]:
# ============================================================================
# 📚 INFORMACIÓN: Ollama Cloud y Servidores Remotos
# ============================================================================

print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                    OLLAMA CLOUD Y SERVIDORES REMOTOS                     ║
╚══════════════════════════════════════════════════════════════════════════╝

🌩️ OLLAMA CLOUD (ollama.com)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Ejecuta modelos grandes sin GPU local. Los modelos se ejecutan en los
servidores de Ollama.

📋 Pasos para configurar:
1. Crea cuenta en ollama.com
2. Genera API key en ollama.com/settings/api
3. En Colab: Secrets (🔑) → Agregar OLLAMA_API_KEY
4. Configura: METHOD = "ollama_cloud"

💡 Modelos disponibles:
   - gpt-oss:120b (modelo grande de propósito general)
   - Más modelos en ollama.com/library

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🖥️ SERVIDOR OLLAMA REMOTO (tu servidor propio)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Conecta a tu propio servidor Ollama (VPS, servidor local, etc.)

📋 Pasos para configurar:
1. En tu servidor: export OLLAMA_HOST=0.0.0.0:11434
2. Inicia Ollama: ollama serve
3. Configura firewall: sudo ufw allow 11434/tcp
4. En Colab: OLLAMA_HOST_REMOTE = "http://tu-ip:11434"
5. Configura: METHOD = "ollama_remote"

💡 Ventajas:
   - Control total sobre el servidor
   - Modelos grandes sin límites de API
   - Privacidad de datos

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 PROMPTS OPTIMIZADOS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Los prompts han sido mejorados con:
✅ Contexto específico de México y violencia de género
✅ Descripciones detalladas con ejemplos
✅ Instrucciones claras paso a paso
✅ Parámetros de generación optimizados (temperature=0.1, etc.)
✅ Manejo robusto de errores

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📖 Documentación completa: Ver archivo OLLAMA_CLOUD_SETUP.md

╚══════════════════════════════════════════════════════════════════════════╝
""")


In [ ]:
# ============================================================================
# Subir archivos de datos
# ============================================================================
# Sube los archivos train.csv, devel.csv, test.csv para cada subtask

from google.colab import files
import os

print("Sube los archivos de datos:")
print("  - subtask1/train.csv")
print("  - subtask1/devel.csv")
print("  - testingkit_extracted/subtask1/test.csv")
print("  - subtask2/train.csv")
print("  - subtask2/devel.csv")
print("  - testingkit_extracted/subtask2/test.csv")
print()

# Crear directorios
os.makedirs("subtask1", exist_ok=True)
os.makedirs("subtask2", exist_ok=True)
os.makedirs("testingkit_extracted/subtask1", exist_ok=True)
os.makedirs("testingkit_extracted/subtask2", exist_ok=True)

# Subir archivos
uploaded = files.upload()

# Mover archivos a sus directorios correspondientes
for filename in uploaded.keys():
    if 'subtask1' in filename and 'test' in filename:
        os.rename(filename, f"testingkit_extracted/subtask1/{os.path.basename(filename)}")
    elif 'subtask2' in filename and 'test' in filename:
        os.rename(filename, f"testingkit_extracted/subtask2/{os.path.basename(filename)}")
    elif 'subtask1' in filename:
        os.rename(filename, f"subtask1/{os.path.basename(filename)}")
    elif 'subtask2' in filename:
        os.rename(filename, f"subtask2/{os.path.basename(filename)}")

print("\n✅ Archivos subidos y organizados")


In [ ]:
# ============================================================================
# EJECUTAR PIPELINE
# ============================================================================
# Este cell ejecuta el pipeline completo de clasificación

# Ejecutar el script principal
# (El código ya está cargado en memoria desde el cell anterior)

print("Iniciando pipeline de clasificación...")
print("="*60)

# El código se ejecutará automáticamente si está en el mismo notebook
# Si no, puedes ejecutar: exec(open('womenhelp2026.py').read())


In [ ]:
# ============================================================================
# DESCARGAR RESULTADOS
# ============================================================================

from google.colab import files
import os

# Descargar archivos de predicción
if os.path.exists("predictions/predictions.zip"):
    files.download("predictions/predictions.zip")
    print("✅ predictions.zip descargado")

if os.path.exists("predictions/subtask1.csv"):
    files.download("predictions/subtask1.csv")
    print("✅ subtask1.csv descargado")

if os.path.exists("predictions/subtask2.csv"):
    files.download("predictions/subtask2.csv")
    print("✅ subtask2.csv descargado")

if os.path.exists("predictions/metrics.json"):
    files.download("predictions/metrics.json")
    print("✅ metrics.json descargado")
    
    # Mostrar métricas
    import json
    with open("predictions/metrics.json", "r") as f:
        metrics = json.load(f)
    print("\nMétricas:")
    print(json.dumps(metrics, indent=2))
